In [ ]:
import sympy as sp
import itertools
from IPython.display import display, Math, Markdown

class GREngine:
    """
    A symbolic General Relativity engine for computing spacetime geometry tensors.

    This class evaluates and caches the fundamental geometric quantities of a
    provided spacetime metric, including Christoffel symbols, the Riemann
    curvature tensor, the Ricci tensor, and the Kretschmann scalar.
    """

    def __init__(self, coords, metric):
        """
        Initializes the geometry engine.

        Args:
            coords (list): A list of SymPy symbols representing the coordinates.
            metric (sp.Matrix): The covariant metric tensor as a SymPy Matrix.
        """
        self.coords = coords
        self.g = metric
        self.n = len(coords)
        self.g_inv = metric.inv()

        self._Gamma = None
        self._Riemann = None
        self._Ricci = None
        self._Kretschmann = None

    def christoffel(self):
        """
        Computes the Christoffel symbols of the second kind.

        Returns:
            sp.MutableDenseNDimArray: A 3D array containing the Christoffel symbols.
        """
        if self._Gamma is not None:
            return self._Gamma

        n, coords, g, g_inv = self.n, self.coords, self.g, self.g_inv
        Gamma = sp.MutableDenseNDimArray.zeros(n, n, n)

        for rho, mu, nu in itertools.product(range(n), repeat=3):
            term = 0
            for sigma in range(n):
                if g_inv[rho, sigma] != 0:
                    term += g_inv[rho, sigma] * (
                            sp.diff(g[sigma, mu], coords[nu]) +
                            sp.diff(g[sigma, nu], coords[mu]) -
                            sp.diff(g[mu, nu], coords[sigma])
                    )
            if term != 0:
                Gamma[rho, mu, nu] = sp.simplify(term / 2)

        self._Gamma = Gamma
        return Gamma

    def riemann(self):
        """
        Computes the Riemann curvature tensor.

        Returns:
            sp.MutableDenseNDimArray: A 4D array containing the Riemann tensor components.
        """
        if self._Riemann is not None:
            return self._Riemann

        Gamma = self.christoffel()
        n, coords = self.n, self.coords
        Riem = sp.MutableDenseNDimArray.zeros(n, n, n, n)

        for rho, sigma, mu, nu in itertools.product(range(n), repeat=4):
            term = sp.diff(Gamma[rho, nu, sigma], coords[mu]) \
                   - sp.diff(Gamma[rho, mu, sigma], coords[nu])
            for lam in range(n):
                term += Gamma[rho, mu, lam] * Gamma[lam, nu, sigma]
                term -= Gamma[rho, nu, lam] * Gamma[lam, mu, sigma]
            if term != 0:
                Riem[rho, sigma, mu, nu] = sp.simplify(term)

        self._Riemann = Riem
        return Riem

    def ricci(self):
        """
        Computes the Ricci tensor by contracting the Riemann tensor.

        Returns:
            sp.MutableDenseMatrix: A 2D matrix containing the Ricci tensor components.
        """
        if self._Ricci is not None:
            return self._Ricci

        Riem = self.riemann()
        n = self.n
        Ric = sp.MutableDenseMatrix.zeros(n, n)

        for mu, nu in itertools.product(range(n), repeat=2):
            term = sum(Riem[rho, mu, rho, nu] for rho in range(n))
            if term != 0:
                Ric[mu, nu] = sp.simplify(term)

        self._Ricci = Ric
        return Ric

    def kretschmann(self):
        """
        Computes the Kretschmann scalar.

        The scalar is found by fully contracting the Riemann tensor with itself.
        Index raising and lowering is performed sequentially prior to contraction.

        Returns:
            sp.Expr: The simplified symbolic expression of the Kretschmann scalar.
        """
        if self._Kretschmann is not None:
            return self._Kretschmann

        R_uddd = self.riemann()
        n = self.n
        g, g_inv = self.g, self.g_inv

        R_dddd = sp.MutableDenseNDimArray.zeros(n, n, n, n)
        for a, b, c, d in itertools.product(range(n), repeat=4):
            term = sum(g[a, e] * R_uddd[e, b, c, d] for e in range(n) if g[a, e] != 0)
            if term != 0: R_dddd[a, b, c, d] = term

        R_uddd_raised = sp.MutableDenseNDimArray.zeros(n, n, n, n)
        for a, b, c, d in itertools.product(range(n), repeat=4):
            term = sum(g_inv[a, e] * R_dddd[e, b, c, d] for e in range(n) if g_inv[a, e] != 0)
            if term != 0: R_uddd_raised[a, b, c, d] = term

        R_uudd = sp.MutableDenseNDimArray.zeros(n, n, n, n)
        for a, b, c, d in itertools.product(range(n), repeat=4):
            term = sum(g_inv[b, e] * R_uddd_raised[a, e, c, d] for e in range(n) if g_inv[b, e] != 0)
            if term != 0: R_uudd[a, b, c, d] = term

        R_uuud = sp.MutableDenseNDimArray.zeros(n, n, n, n)
        for a, b, c, d in itertools.product(range(n), repeat=4):
            term = sum(g_inv[c, e] * R_uudd[a, b, e, d] for e in range(n) if g_inv[c, e] != 0)
            if term != 0: R_uuud[a, b, c, d] = term

        R_uuuu = sp.MutableDenseNDimArray.zeros(n, n, n, n)
        for a, b, c, d in itertools.product(range(n), repeat=4):
            term = sum(g_inv[d, e] * R_uuud[a, b, c, e] for e in range(n) if g_inv[d, e] != 0)
            if term != 0: R_uuuu[a, b, c, d] = term

        K = sum(R_dddd[a, b, c, d] * R_uuuu[a, b, c, d]
                for a, b, c, d in itertools.product(range(n), repeat=4))

        self._Kretschmann = sp.simplify(K)
        return self._Kretschmann



def derive_schwarzschild_metric():
    """
    Executes the symbolic derivation and validation of the Schwarzschild metric.
    """
    # FIX 1: Just spell out the Greek letters. SymPy will auto-render them.
    t, r, th, ph = sp.symbols('t r theta phi', real=True)
    coords = [t, r, th, ph]
    alpha, beta = sp.Function('alpha')(r), sp.Function('beta')(r)

    g_ansatz = sp.diag(-sp.exp(2 * alpha), sp.exp(2 * beta), r ** 2, r ** 2 * sp.sin(th) ** 2)
    engine = GREngine(coords, g_ansatz)

    display(Markdown(r"**GENERAL TENSORS WRT FUNCTIONS α(r) AND β(r)**"))
    display(Markdown(r"*General Christoffel Symbols*"))
    print("\n")

    Gamma_gen = engine.christoffel()
    for i, j, k in itertools.product(range(4), repeat=3):
        if Gamma_gen[i, j, k] != 0:
            # FIX 2: Generate a LaTeX string and display it as Math
            latex_str = rf"\Gamma^{{{sp.latex(coords[i])}}}_{{{sp.latex(coords[j])} {sp.latex(coords[k])}}} = {sp.latex(Gamma_gen[i, j, k])}"
            display(Math(latex_str))
            print("\n")

    print("\n ---------------------------------------------- \n\n")
    display(Markdown(r"*General Riemann Curvature Tensor Components*"))
    print("\n")

    Riem_gen = engine.riemann()
    for i, j, k, l in itertools.product(range(4), repeat=4):
        if Riem_gen[i, j, k, l] != 0:
            latex_str = rf"R^{{{sp.latex(coords[i])}}}_{{\ \ {sp.latex(coords[j])} {sp.latex(coords[k])} {sp.latex(coords[l])}}} = {sp.latex(Riem_gen[i, j, k, l])}"
            display(Math(latex_str))
            print("\n")

    print("\n ---------------------------------------------- \n\n")
    display(Markdown(r"*General Ricci Tensor Components*"))
    print("\n")

    Ric = engine.ricci()
    for i, j in itertools.product(range(4), repeat=2):
        if Ric[i, j] != 0:
            latex_str = rf"R_{{{sp.latex(coords[i])} {sp.latex(coords[j])}}} ="
            display(Math(latex_str))
            display(Ric[i, j])
            print("\n")

    print("\n\n ---------------------------------------------- \n")
    display(Markdown(r"**Step 1:** Solving $R^t_t - R^r_r = 0$"))

    eq_alpha_beta = sp.simplify(engine.g_inv[0, 0] * Ric[0, 0] - engine.g_inv[1, 1] * Ric[1, 1])
    g_inv = engine.g_inv
    display(Math(rf"\text{{Relationship eq: }} {sp.latex(eq_alpha_beta)} = 0"))
    display(Markdown(r"$\Rightarrow \alpha'(r) = -\beta'(r) \Rightarrow \beta(r) = -\alpha(r)$ *(setting integration constant to 0 for asymptotic flatness)*"))
    print("\n")
    display(Markdown(r"**Step 2:** Solving $R_{\theta\theta} = 0$ to find $\alpha(r)$"))

    y = sp.Function('y')(r)

    R_thth_sub = Ric[2, 2].subs(beta, -alpha).subs(alpha, sp.log(y) / 2).doit()
    expr_y = sp.simplify(R_thth_sub)
    display(Math(rf"\text{{ODE for }} y(r) = e^{{2\alpha}}: {sp.latex(expr_y)} = 0"))

    sol = sp.dsolve(sp.Eq(expr_y, 0), y)
    display(Math(rf"\text{{General solution for }} y(r): {sp.latex(sol.rhs)}"))

    constants = list(sol.rhs.free_symbols - {r})
    if constants:
        C1 = constants[0]
        rs = sp.symbols('r_s', real=True)
        y_final = sol.rhs.subs(C1, -rs)
        display(Math(rf"\text{{Applying physical boundary conditions (constant }} = -r_s): y(r) = {sp.latex(y_final)}"))
        print()
    else:
        raise ValueError("SymPy did not return a constant of integration.")

    print("\n")
    display(Markdown(r"**Step 3: Constructing and Validating the Derived Metric**"))

    alpha_final = sp.log(y_final) / 2
    beta_final = -alpha_final

    g_derived = sp.simplify(sp.diag(-sp.exp(2 * alpha_final), sp.exp(2 * beta_final), r ** 2, r ** 2 * sp.sin(th) ** 2))

    val_engine = GREngine(coords, g_derived)
    ricci_final = val_engine.ricci()

    is_vacuum = True
    for i, j in itertools.product(range(4), repeat=2):
        if sp.simplify(ricci_final[i, j]) != 0:
            is_vacuum = False

    display(Markdown(rf"**Einstein Vacuum Equations** ($R_{{\mu\nu}} = 0$) satisfied dynamically: **{is_vacuum}**"))
    display(Markdown(r"<br>**Line Element** ($ds^2$):"))

    dt, dr, dth, dph = sp.symbols('dt dr dtheta dphi')
    ds2 = g_derived[0, 0] * dt ** 2 + g_derived[1, 1] * dr ** 2 + g_derived[2, 2] * dth ** 2 + g_derived[3, 3] * dph ** 2
    display(sp.Eq(sp.Symbol('ds^2'), ds2))

    print("\n---------------------------------------------------\n")

    display(Markdown(r"**Kretschmann Scalar** ($K = R_{abcd} R^{abcd}$):"))
    k_scalar = val_engine.kretschmann()
    display(k_scalar)

    print("\n---------------------------------------------------\n")
    display(Markdown("*Non-zero Christoffel Symbols (Connections)*"))
    print("\n")

    Gamma_final = val_engine.christoffel()
    for i, j, k in itertools.product(range(4), repeat=3):
        if Gamma_final[i, j, k] != 0:
            latex_str = rf"\Gamma^{{{sp.latex(coords[i])}}}_{{{sp.latex(coords[j])} {sp.latex(coords[k])}}} = {sp.latex(Gamma_final[i, j, k])}"
            display(Math(latex_str))
            print("\n")

    print("---------------------------------------------------\n")

    display(Markdown(r"<br>**Non-zero Riemann Curvature Tensor Components** ($R^\rho_{\ \sigma\mu\nu}$)"))
    Riem_final = val_engine.riemann()
    for i, j, k, l in itertools.product(range(4), repeat=4):
        if Riem_final[i, j, k, l] != 0:
            latex_str = rf"R^{{{sp.latex(coords[i])}}}_{{\ \ {sp.latex(coords[j])} {sp.latex(coords[k])} {sp.latex(coords[l])}}} = {sp.latex(Riem_final[i, j, k, l])}"
            display(Math(latex_str))
            print("\n")

    print("---------------------------------------------------\n")


if __name__ == "__main__":
    derive_schwarzschild_metric()



**GENERAL TENSORS WRT FUNCTIONS α(r) AND β(r)**

*General Christoffel Symbols*

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>



---------------------------------------------------



In [ ]:
def demonstrate_birkhoff():
    """
    Executes the symbolic derivation of Birkhoff's Theorem, mirroring
    the exact same tensor evaluation and validation structure of the
    Schwarzschild metric derivation.
    """
    t, r, th, ph = sp.symbols('t r theta phi', real=True)
    coords = [t, r, th, ph]

    # EXACT SAME, but functions depend on BOTH t and r
    alpha, beta = sp.Function('alpha')(t, r), sp.Function('beta')(t, r)

    g_ansatz = sp.diag(-sp.exp(2 * alpha), sp.exp(2 * beta), r ** 2, r ** 2 * sp.sin(th) ** 2)
    engine = GREngine(coords, g_ansatz)

    display(Markdown(r"**GENERAL TENSORS WRT FUNCTIONS α(t,r) AND β(t,r)**"))
    display(Markdown(r"*General Christoffel Symbols*"))
    print("\n")

    Gamma_gen = engine.christoffel()
    for i, j, k in itertools.product(range(4), repeat=3):
        if Gamma_gen[i, j, k] != 0:
            latex_str = rf"\Gamma^{{{sp.latex(coords[i])}}}_{{{sp.latex(coords[j])} {sp.latex(coords[k])}}} = {sp.latex(Gamma_gen[i, j, k])}"
            display(Math(latex_str))
            print("\n")

    print("\n ---------------------------------------------- \n\n")
    display(Markdown(r"*General Riemann Curvature Tensor Components*"))
    print("\n")

    Riem_gen = engine.riemann()
    for i, j, k, l in itertools.product(range(4), repeat=4):
        if Riem_gen[i, j, k, l] != 0:
            latex_str = rf"R^{{{sp.latex(coords[i])}}}_{{\ \ {sp.latex(coords[j])} {sp.latex(coords[k])} {sp.latex(coords[l])}}} = {sp.latex(Riem_gen[i, j, k, l])}"
            display(Math(latex_str))
            print("\n")

    print("\n ---------------------------------------------- \n\n")
    display(Markdown(r"*General Ricci Tensor Components*"))
    print("\n")

    Ric = engine.ricci()
    for i, j in itertools.product(range(4), repeat=2):
        if Ric[i, j] != 0:
            latex_str = rf"R_{{{sp.latex(coords[i])} {sp.latex(coords[j])}}} ="
            display(Math(latex_str))
            display(Ric[i, j])
            print("\n")

    print("\n\n ---------------------------------------------- \n")
    display(((engine.g_inv)))

    display(Markdown(r"**Step 1:** Solving $R_{tr} = 0$ (Birkhoff's first constraint)"))

    eq_tr = sp.simplify(Ric[0, 1])
    display(Math(rf"\text{{Relationship eq: }} {sp.latex(eq_tr)} = 0"))
    display(Markdown(r"$\Rightarrow \frac{\partial}{\partial t}\beta(t, r) = 0 \Rightarrow \beta(t, r) = \beta(r)$"))
    print("\n")

    display(Markdown(r"**Step 2:** Solving $R^t_t - R^r_r = 0$ (Applying static $\beta$)"))

    # Substitute beta(t,r) with static beta(r) to prevent SymPy PDE issues
    beta_r = sp.Function('beta')(r)
    alpha_r = sp.Function('alpha')(r)

    eq_alpha_beta = sp.simplify((engine.g_inv[0, 0] * Ric[0, 0] - engine.g_inv[1, 1] * Ric[1, 1]).subs(beta, beta_r).doit())
    display(Math(rf"\text{{Relationship eq: }} {sp.latex(eq_alpha_beta)} = 0"))
    display(Markdown(r"$\Rightarrow \frac{\partial}{\partial r}\alpha(t, r) = -\frac{d}{dr}\beta(r) \Rightarrow \alpha(t, r) = -\beta(r) + f(t)$"))
    display(Markdown(r"$\Rightarrow$ Absorbing $f(t)$ into time coordinate $dt' = e^{f(t)}dt$ gives static $\alpha(r) = -\beta(r)$"))
    print("\n")

    display(Markdown(r"**Step 3:** Solving $R_{\theta\theta} = 0$ to find $\alpha(r)$"))
    y = sp.Function('y')(r)

    Ric_static = Ric[2, 2].subs({alpha: alpha_r, beta: beta_r}).doit()
    R_thth_sub = Ric_static.subs(beta_r, -alpha_r).subs(alpha_r, sp.log(y) / 2).doit()

    expr_y = sp.simplify(R_thth_sub)
    display(Math(rf"\text{{ODE for }} y(r) = e^{{2\alpha}}: {sp.latex(expr_y)} = 0"))

    sol = sp.dsolve(sp.Eq(expr_y, 0), y)
    display(Math(rf"\text{{General solution for }} y(r): {sp.latex(sol.rhs)}"))

    constants = list(sol.rhs.free_symbols - {r})
    if constants:
        C1 = constants[0]
        rs = sp.symbols('r_s', real=True)
        y_final = sol.rhs.subs(C1, -rs)
        display(Math(rf"\text{{Applying physical boundary conditions (constant }} = -r_s): y(r) = {sp.latex(y_final)}"))
        print()
    else:
        raise ValueError("SymPy did not return a constant of integration.")

    print("\n")
    display(Markdown(r"**Step 4: Constructing and Validating the Derived Metric**"))

    alpha_final = sp.log(y_final) / 2
    beta_final = -alpha_final

    g_derived = sp.simplify(sp.diag(-sp.exp(2 * alpha_final), sp.exp(2 * beta_final), r ** 2, r ** 2 * sp.sin(th) ** 2))

    val_engine = GREngine(coords, g_derived)
    ricci_final = val_engine.ricci()

    is_vacuum = True
    for i, j in itertools.product(range(4), repeat=2):
        if sp.simplify(ricci_final[i, j]) != 0:
            is_vacuum = False

    display(Markdown(rf"**Einstein Vacuum Equations** ($R_{{\mu\nu}} = 0$) satisfied dynamically: **{is_vacuum}**"))
    display(Markdown(r"<br>**Line Element** ($ds^2$):"))

    dt, dr, dth, dph = sp.symbols(r'dt dr d\theta d\phi')
    ds2 = g_derived[0, 0] * dt ** 2 + g_derived[1, 1] * dr ** 2 + g_derived[2, 2] * dth ** 2 + g_derived[3, 3] * dph ** 2
    display(sp.Eq(sp.Symbol('ds^2'), ds2))

    print("\n---------------------------------------------------\n")

    display(Markdown(r"**Kretschmann Scalar** ($K = R_{abcd} R^{abcd}$):"))
    k_scalar = val_engine.kretschmann()
    display(k_scalar)

    print("\n---------------------------------------------------\n")
    display(Markdown("*Non-zero Christoffel Symbols (Connections)*"))
    print("\n")

    Gamma_final = val_engine.christoffel()
    for i, j, k in itertools.product(range(4), repeat=3):
        if Gamma_final[i, j, k] != 0:
            latex_str = rf"\Gamma^{{{sp.latex(coords[i])}}}_{{{sp.latex(coords[j])} {sp.latex(coords[k])}}} = {sp.latex(Gamma_final[i, j, k])}"
            display(Math(latex_str))
            print("\n")

    print("---------------------------------------------------\n")

    display(Markdown(r"<br>**Non-zero Riemann Curvature Tensor Components** ($R^\rho_{\ \sigma\mu\nu}$)"))
    Riem_final = val_engine.riemann()
    for i, j, k, l in itertools.product(range(4), repeat=4):
        if Riem_final[i, j, k, l] != 0:
            latex_str = rf"R^{{{sp.latex(coords[i])}}}_{{\ \ {sp.latex(coords[j])} {sp.latex(coords[k])} {sp.latex(coords[l])}}} = {sp.latex(Riem_final[i, j, k, l])}"
            display(Math(latex_str))
            print("\n")

    print("---------------------------------------------------\n")

    display(Markdown(r"<br>**Ricci Tensor Components** ($R_{\mu\nu}$)"))
    ricci_nonzero = False
    for i, j in itertools.product(range(4), repeat=2):
        if ricci_final[i, j] == 0:
            latex_str = rf"R_{{{sp.latex(coords[i])} {sp.latex(coords[j])}}} = {sp.latex(ricci_final[i, j])}"
            display(Math(latex_str))
            ricci_nonzero = True

    if not ricci_nonzero:
        display(Markdown("*All Ricci tensor components are identically zero (Vacuum solution confirmed).*"))

if __name__ == "__main__":
  demonstrate_birkhoff()

**GENERAL TENSORS WRT FUNCTIONS α(t,r) AND β(t,r)**

*General Christoffel Symbols*

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>